# CUDA Cluster-Pool Sparse FFN Notebook

This notebook does **not** require old training checkpoints by default. The default path is a synthetic CUDA benchmark for the new hardware shape:

```text
selector features → token clusters → shared candidate pool per cluster → cluster GEMMs
```

If you do have a dense checkpoint on the remote Colab filesystem, set `RUN_CHECKPOINT_ORACLE = True` and point `DENSE_CHECKPOINT` at it. Otherwise the checkpoint cells cleanly skip.

In [ ]:
from __future__ import annotations

import dataclasses
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any


def _run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


def find_or_clone_repo() -> Path:
    env_root = os.environ.get("SPEEDUP_THINGY_REPO")
    candidates: list[Path] = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd.parent,
        Path("/content/Speedup-Thingy"),
        Path("/workspace/Speedup-Thingy"),
        Path.home() / "Speedup-Thingy",
    ])
    for candidate in candidates:
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    clone_dir = Path(os.environ.get("SPEEDUP_THINGY_CLONE_DIR", "/content/Speedup-Thingy"))
    repo_url = os.environ.get("SPEEDUP_THINGY_REPO_URL", "https://github.com/BrownHujay/Speedup-Thingy.git")
    repo_ref = os.environ.get("SPEEDUP_THINGY_REF", "master")
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    if not clone_dir.exists():
        _run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(clone_dir)])
    elif (clone_dir / ".git").exists():
        _run(["git", "fetch", "origin", repo_ref], cwd=clone_dir)
        _run(["git", "checkout", repo_ref], cwd=clone_dir)
        _run(["git", "pull", "--ff-only"], cwd=clone_dir)
    return clone_dir.resolve()


repo_root = find_or_clone_repo()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    import recursive_training_engine  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"])

import torch
import torch.nn.functional as F

try:
    from recursive_training_engine.config import load_config
    from recursive_training_engine.data import load_token_streams
    from recursive_training_engine.models import DenseModel
    from recursive_training_engine.utils import set_seed
    HAVE_REPO_MODEL_CODE = True
except Exception as exc:
    print("repo model imports failed; synthetic benchmark still works:", repr(exc))
    HAVE_REPO_MODEL_CODE = False

print("repo_root:", repo_root)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

## Settings

Default is safe and checkpoint-free. It benchmarks the cluster-pool FFN shape directly on random tensors. This is the systems test we need after the per-token sparse kernel failed.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

# Synthetic FFN benchmark. This needs no checkpoint and is the default thing to run.
D_MODEL = 2048
D_FF = 8192
TOKENS = 4096
RANK = 64
CLUSTERS = [8, 16, 32, 64]
CANDIDATE_M = [96, 128, 192, 256]
CLUSTER_ITERS = 4
DTYPE = torch.float16
WARMUP = 3
ITERS = 10
SEED = 123

# Optional checkpoint-quality oracle. Off by default because checkpoints are not in the GitHub clone.
RUN_CHECKPOINT_ORACLE = False
CONFIG = repo_root / "configs/proof_filter_depth4_dense.yaml"
DENSE_CHECKPOINT = Path(os.environ.get("DENSE_CHECKPOINT", ""))
if not str(DENSE_CHECKPOINT):
    DENSE_CHECKPOINT = repo_root / "runs/proof_d4_dense/checkpoint.pt"
EVAL_BATCHES: int | None = 1
BATCH_SIZE: int | None = None
REFERENCE_K = 64
TEMPERATURE = 2.0
COMPUTE_RECALL = True

print(json.dumps({
    "synthetic": {
        "d_model": D_MODEL,
        "d_ff": D_FF,
        "tokens": TOKENS,
        "rank": RANK,
        "clusters": CLUSTERS,
        "candidate_m": CANDIDATE_M,
        "dtype": str(DTYPE),
    },
    "checkpoint_oracle": {
        "enabled": RUN_CHECKPOINT_ORACLE,
        "config": str(CONFIG),
        "dense_checkpoint": str(DENSE_CHECKPOINT),
        "checkpoint_exists": DENSE_CHECKPOINT.exists(),
    },
    "device": str(DEVICE),
}, indent=2))

## Shared Cluster-Pool Helpers

In [ ]:
def cuda_time_ms(fn, *, warmup: int = WARMUP, iters: int = ITERS) -> float:
    for _ in range(warmup):
        fn()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(iters):
            fn()
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / iters
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    return (time.perf_counter() - t0) * 1000.0 / iters


def cluster_assignments_from_features(features: torch.Tensor, *, cluster_count: int, iters: int) -> torch.Tensor:
    token_count = features.shape[0]
    cluster_count = min(max(1, int(cluster_count)), max(1, token_count))
    if cluster_count == 1 or token_count == 1:
        return torch.zeros(token_count, device=features.device, dtype=torch.long)
    x = F.normalize(features.float(), dim=-1)
    init_idx = torch.linspace(0, token_count - 1, steps=cluster_count, device=features.device).round().long()
    centers = x.index_select(0, init_idx).contiguous()
    assignments = torch.zeros(token_count, device=features.device, dtype=torch.long)
    for _ in range(max(1, int(iters))):
        assignments = (x @ centers.t()).argmax(dim=-1)
        new_centers = torch.zeros_like(centers)
        counts = torch.bincount(assignments, minlength=cluster_count).to(new_centers.dtype)
        new_centers.index_add_(0, assignments, x)
        nonempty = counts > 0
        new_centers[nonempty] = new_centers[nonempty] / counts[nonempty].unsqueeze(-1).clamp_min(1.0)
        new_centers[~nonempty] = centers[~nonempty]
        centers = F.normalize(new_centers, dim=-1)
    return assignments


def aggregate_cluster_scores(scores: torch.Tensor, assignments: torch.Tensor, *, cluster_count: int) -> torch.Tensor:
    agg = scores.new_zeros(cluster_count, scores.shape[-1])
    agg.index_add_(0, assignments, scores)
    counts = torch.bincount(assignments, minlength=cluster_count).to(scores.dtype).clamp_min(1.0)
    return agg / counts.unsqueeze(-1)



def kmeans_assignments_and_centers(features: torch.Tensor, *, cluster_count: int, iters: int):
    token_count = features.shape[0]
    cluster_count = min(max(1, int(cluster_count)), max(1, token_count))
    x_norm = F.normalize(features.float(), dim=-1)
    if cluster_count == 1 or token_count == 1:
        assignments = torch.zeros(token_count, device=features.device, dtype=torch.long)
        centers = F.normalize(x_norm.mean(dim=0, keepdim=True), dim=-1)
        return assignments, centers
    init_idx = torch.linspace(0, token_count - 1, steps=cluster_count, device=features.device).round().long()
    centers = x_norm.index_select(0, init_idx).contiguous()
    assignments = torch.zeros(token_count, device=features.device, dtype=torch.long)
    for _ in range(max(1, int(iters))):
        assignments = (x_norm @ centers.t()).argmax(dim=-1)
        new_centers = torch.zeros_like(centers)
        counts = torch.bincount(assignments, minlength=cluster_count).to(new_centers.dtype)
        new_centers.index_add_(0, assignments, x_norm)
        nonempty = counts > 0
        new_centers[nonempty] = new_centers[nonempty] / counts[nonempty].unsqueeze(-1).clamp_min(1.0)
        new_centers[~nonempty] = centers[~nonempty]
        centers = F.normalize(new_centers, dim=-1)
    return assignments, centers


def route_from_static_centers(x, up_a, gate_a, centers):
    q_up = x @ up_a
    q_gate = x @ gate_a
    features = F.normalize(torch.cat([q_up, q_gate], dim=-1).float(), dim=-1)
    return (features @ centers.t()).argmax(dim=-1)


def build_static_cluster_plan(x, w_down, up_a, up_b, gate_a, gate_b, *, clusters: int, candidate_m: int):
    q_up = x @ up_a
    q_gate = x @ gate_a
    up_hat = q_up @ up_b
    gate_hat = q_gate @ gate_b
    gate_act = F.silu(gate_hat)
    wd_norm = w_down.norm(dim=1).to(x.dtype)
    scores = (up_hat.abs() + gate_act.abs() + (up_hat * gate_act).abs()) * wd_norm
    assignments, centers = kmeans_assignments_and_centers(
        torch.cat([q_up, q_gate], dim=-1),
        cluster_count=clusters,
        iters=CLUSTER_ITERS,
    )
    cluster_count = min(clusters, x.shape[0])
    pooled = aggregate_cluster_scores(scores.float(), assignments, cluster_count=cluster_count)
    candidate_ids = torch.topk(pooled, k=min(candidate_m, w_down.shape[0]), dim=-1).indices
    return assignments, centers, candidate_ids, torch.bincount(assignments, minlength=cluster_count)

def build_cluster_plan(x, w_down, up_a, up_b, gate_a, gate_b, *, clusters: int, candidate_m: int):
    q_up = x @ up_a
    q_gate = x @ gate_a
    up_hat = q_up @ up_b
    gate_hat = q_gate @ gate_b
    gate_act = F.silu(gate_hat)
    wd_norm = w_down.norm(dim=1).to(x.dtype)
    scores = (up_hat.abs() + gate_act.abs() + (up_hat * gate_act).abs()) * wd_norm
    assignments = cluster_assignments_from_features(
        torch.cat([q_up, q_gate], dim=-1),
        cluster_count=clusters,
        iters=CLUSTER_ITERS,
    )
    cluster_count = min(clusters, x.shape[0])
    pooled = aggregate_cluster_scores(scores.float(), assignments, cluster_count=cluster_count)
    candidate_ids = torch.topk(pooled, k=min(candidate_m, w_down.shape[0]), dim=-1).indices
    cluster_sizes = torch.bincount(assignments, minlength=cluster_count)
    return assignments, candidate_ids, cluster_sizes


def cluster_execute_loop(x, w_up, w_gate, w_down, assignments, candidate_ids):
    out = x.new_zeros(x.shape[0], x.shape[1])
    for cluster_idx in range(candidate_ids.shape[0]):
        token_idx = torch.nonzero(assignments == cluster_idx, as_tuple=False).flatten()
        if token_idx.numel() == 0:
            continue
        ids = candidate_ids[cluster_idx]
        xc = x.index_select(0, token_idx)
        up = xc @ w_up.index_select(0, ids).t().contiguous()
        gate = xc @ w_gate.index_select(0, ids).t().contiguous()
        z = up * F.silu(gate)
        out.index_copy_(0, token_idx, z @ w_down.index_select(0, ids).contiguous())
    return out


def build_prepacked_cluster_tensors(x, w_up, w_gate, w_down, assignments, candidate_ids):
    cluster_count = candidate_ids.shape[0]
    counts = torch.bincount(assignments, minlength=cluster_count)
    max_count = int(counts.max().item()) if counts.numel() else 0
    x_pad = x.new_zeros(cluster_count, max_count, x.shape[1])
    token_index_pad = torch.full((cluster_count, max_count), -1, device=x.device, dtype=torch.long)
    for cluster_idx in range(cluster_count):
        token_idx = torch.nonzero(assignments == cluster_idx, as_tuple=False).flatten()
        n = token_idx.numel()
        if n:
            x_pad[cluster_idx, :n] = x.index_select(0, token_idx)
            token_index_pad[cluster_idx, :n] = token_idx
    wup_pool = w_up[candidate_ids].transpose(1, 2).contiguous()
    wgate_pool = w_gate[candidate_ids].transpose(1, 2).contiguous()
    wdown_pool = w_down[candidate_ids].contiguous()
    return x_pad, wup_pool, wgate_pool, wdown_pool, token_index_pad, counts



def pack_tokens_by_assignment(x, assignments, *, cluster_count: int, max_count: int):
    token_count = x.shape[0]
    one_hot = F.one_hot(assignments, num_classes=cluster_count).to(torch.int32)
    positions_all = one_hot.cumsum(dim=0) - 1
    positions = positions_all[torch.arange(token_count, device=x.device), assignments].long()
    x_pad = x.new_zeros(cluster_count, max_count, x.shape[-1])
    x_pad[assignments, positions] = x
    return x_pad, positions


def cluster_execute_pack_bmm_scatter(x, assignments, wup_pool, wgate_pool, wdown_pool, *, max_count: int):
    x_pad, positions = pack_tokens_by_assignment(
        x,
        assignments,
        cluster_count=wup_pool.shape[0],
        max_count=max_count,
    )
    y_pad = cluster_execute_prepacked_bmm(x_pad, wup_pool, wgate_pool, wdown_pool)
    return y_pad[assignments, positions]


def build_static_pack_gather_indices(assignments: torch.Tensor, *, cluster_count: int):
    counts = torch.bincount(assignments, minlength=cluster_count)
    max_count = int(counts.max().item()) if counts.numel() else 0
    pack_index = torch.zeros(cluster_count, max_count, device=assignments.device, dtype=torch.long)
    flat_gather = torch.empty(assignments.shape[0], device=assignments.device, dtype=torch.long)
    for cluster_idx in range(cluster_count):
        token_idx = torch.nonzero(assignments == cluster_idx, as_tuple=False).flatten()
        n = token_idx.numel()
        if n:
            pack_index[cluster_idx, :n] = token_idx
            flat_gather[token_idx] = cluster_idx * max_count + torch.arange(n, device=assignments.device)
    return pack_index, flat_gather, counts, max_count


def cluster_execute_preindexed_pack_bmm_gather(x, pack_index, flat_gather, wup_pool, wgate_pool, wdown_pool):
    cluster_count, max_count = pack_index.shape
    x_pad = x.index_select(0, pack_index.reshape(-1)).view(cluster_count, max_count, x.shape[-1])
    y_pad = cluster_execute_prepacked_bmm(x_pad, wup_pool, wgate_pool, wdown_pool)
    return y_pad.reshape(cluster_count * max_count, x.shape[-1]).index_select(0, flat_gather)

def cluster_execute_prepacked_bmm(x_pad, wup_pool, wgate_pool, wdown_pool):
    up = torch.bmm(x_pad, wup_pool)
    gate = torch.bmm(x_pad, wgate_pool)
    z = up * F.silu(gate)
    return torch.bmm(z, wdown_pool)

## Synthetic Cluster-Pool Benchmark

This is the main default run. It needs no old checkpoint. It compares dense SwiGLU to cluster-shared candidate-pool GEMMs on the same random FFN weights.

In [ ]:
torch.manual_seed(SEED)
x = torch.randn(TOKENS, D_MODEL, device=DEVICE, dtype=DTYPE)
scale = 1.0 / math.sqrt(D_MODEL)
w_up = torch.randn(D_FF, D_MODEL, device=DEVICE, dtype=DTYPE) * scale
w_gate = torch.randn(D_FF, D_MODEL, device=DEVICE, dtype=DTYPE) * scale
w_down = torch.randn(D_FF, D_MODEL, device=DEVICE, dtype=DTYPE) * scale
up_a = torch.randn(D_MODEL, RANK, device=DEVICE, dtype=DTYPE) * scale
up_b = torch.randn(RANK, D_FF, device=DEVICE, dtype=DTYPE) * scale
gate_a = torch.randn(D_MODEL, RANK, device=DEVICE, dtype=DTYPE) * scale
gate_b = torch.randn(RANK, D_FF, device=DEVICE, dtype=DTYPE) * scale


def dense_ffn():
    up = x @ w_up.t()
    gate = x @ w_gate.t()
    return (up * F.silu(gate)) @ w_down

# Warm dense once before sparse variants.
dense_ffn()
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
dense_ms = cuda_time_ms(dense_ffn)
print("dense_ms", dense_ms)

rows = []
for clusters in CLUSTERS:
    for candidate_m in CANDIDATE_M:
        def plan_only(clusters=clusters, candidate_m=candidate_m):
            return build_cluster_plan(x, w_down, up_a, up_b, gate_a, gate_b, clusters=clusters, candidate_m=candidate_m)

        assignments, candidate_ids, cluster_sizes = plan_only()
        static_assignments, static_centers, static_candidate_ids, static_cluster_sizes = build_static_cluster_plan(
            x, w_down, up_a, up_b, gate_a, gate_b, clusters=clusters, candidate_m=candidate_m
        )
        x_pad, wup_pool, wgate_pool, wdown_pool, token_index_pad, counts = build_prepacked_cluster_tensors(
            x, w_up, w_gate, w_down, assignments, candidate_ids
        )
        static_x_pad, static_wup_pool, static_wgate_pool, static_wdown_pool, static_token_index_pad, static_counts = build_prepacked_cluster_tensors(
            x, w_up, w_gate, w_down, static_assignments, static_candidate_ids
        )

        def loop_exec():
            return cluster_execute_loop(x, w_up, w_gate, w_down, assignments, candidate_ids)

        def prepacked_bmm_exec():
            return cluster_execute_prepacked_bmm(x_pad, wup_pool, wgate_pool, wdown_pool)

        def static_route_only():
            return route_from_static_centers(x, up_a, gate_a, static_centers)

        static_max_count = int(static_counts.max().item()) if static_counts.numel() else 0
        static_pack_index, static_flat_gather, _, _ = build_static_pack_gather_indices(
            static_assignments,
            cluster_count=static_candidate_ids.shape[0],
        )

        def static_prepacked_bmm_exec():
            return cluster_execute_prepacked_bmm(static_x_pad, static_wup_pool, static_wgate_pool, static_wdown_pool)

        def pack_bmm_scatter_exec():
            return cluster_execute_pack_bmm_scatter(
                x,
                static_assignments,
                static_wup_pool,
                static_wgate_pool,
                static_wdown_pool,
                max_count=static_max_count,
            )

        def preindexed_pack_bmm_gather_exec():
            return cluster_execute_preindexed_pack_bmm_gather(
                x,
                static_pack_index,
                static_flat_gather,
                static_wup_pool,
                static_wgate_pool,
                static_wdown_pool,
            )

        def static_route_plus_pack_bmm_scatter():
            a = route_from_static_centers(x, up_a, gate_a, static_centers)
            return cluster_execute_pack_bmm_scatter(
                x,
                a,
                static_wup_pool,
                static_wgate_pool,
                static_wdown_pool,
                max_count=static_max_count,
            )

        def static_route_plus_loop():
            a = route_from_static_centers(x, up_a, gate_a, static_centers)
            return cluster_execute_loop(x, w_up, w_gate, w_down, a, static_candidate_ids)

        def full_plan_plus_loop():
            a, ids, _ = build_cluster_plan(x, w_down, up_a, up_b, gate_a, gate_b, clusters=clusters, candidate_m=candidate_m)
            return cluster_execute_loop(x, w_up, w_gate, w_down, a, ids)

        # One correctness sanity check for the two execution paths.
        y_loop = loop_exec()
        y_pack = prepacked_bmm_exec()
        y_preindexed = preindexed_pack_bmm_gather_exec()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        valid_abs = []
        for c in range(candidate_ids.shape[0]):
            idx = token_index_pad[c]
            mask = idx >= 0
            if bool(mask.any()):
                valid_abs.append((y_loop.index_select(0, idx[mask]) - y_pack[c, mask]).abs().max())
        max_exec_diff = float(torch.stack(valid_abs).max().detach().cpu()) if valid_abs else 0.0
        max_preindexed_diff = float((cluster_execute_loop(x, w_up, w_gate, w_down, static_assignments, static_candidate_ids) - y_preindexed).abs().max().detach().cpu())

        plan_ms = cuda_time_ms(plan_only, warmup=2, iters=5)
        loop_exec_ms = cuda_time_ms(loop_exec, warmup=2, iters=5)
        prepacked_bmm_ms = cuda_time_ms(prepacked_bmm_exec, warmup=3, iters=10)
        static_route_ms = cuda_time_ms(static_route_only, warmup=3, iters=10)
        static_prepacked_bmm_ms = cuda_time_ms(static_prepacked_bmm_exec, warmup=3, iters=10)
        pack_bmm_scatter_ms = cuda_time_ms(pack_bmm_scatter_exec, warmup=3, iters=10)
        preindexed_pack_bmm_gather_ms = cuda_time_ms(preindexed_pack_bmm_gather_exec, warmup=3, iters=10)
        static_route_pack_bmm_scatter_ms = cuda_time_ms(static_route_plus_pack_bmm_scatter, warmup=3, iters=10)
        static_route_plus_loop_ms = cuda_time_ms(static_route_plus_loop, warmup=2, iters=5)
        full_ms = cuda_time_ms(full_plan_plus_loop, warmup=1, iters=3)

        nonempty = cluster_sizes[cluster_sizes > 0].float()
        row = {
            "clusters": clusters,
            "candidate_m": candidate_m,
            "dense_ms": dense_ms,
            "plan_ms": plan_ms,
            "loop_exec_ms": loop_exec_ms,
            "prepacked_bmm_exec_ms": prepacked_bmm_ms,
            "static_route_ms": static_route_ms,
            "static_prepacked_bmm_exec_ms": static_prepacked_bmm_ms,
            "pack_bmm_scatter_ms": pack_bmm_scatter_ms,
            "preindexed_pack_bmm_gather_ms": preindexed_pack_bmm_gather_ms,
            "static_route_pack_bmm_scatter_ms": static_route_pack_bmm_scatter_ms,
            "static_route_plus_loop_ms": static_route_plus_loop_ms,
            "static_route_plus_prepacked_lower_bound_ms": static_route_ms + static_prepacked_bmm_ms,
            "full_plan_plus_loop_ms": full_ms,
            "speedup_full_plan_plus_loop": dense_ms / max(full_ms, 1e-12),
            "speedup_loop_exec_only": dense_ms / max(loop_exec_ms, 1e-12),
            "speedup_prepacked_bmm_only": dense_ms / max(prepacked_bmm_ms, 1e-12),
            "speedup_pack_bmm_scatter": dense_ms / max(pack_bmm_scatter_ms, 1e-12),
            "speedup_preindexed_pack_bmm_gather": dense_ms / max(preindexed_pack_bmm_gather_ms, 1e-12),
            "speedup_static_route_pack_bmm_scatter": dense_ms / max(static_route_pack_bmm_scatter_ms, 1e-12),
            "speedup_static_route_plus_loop": dense_ms / max(static_route_plus_loop_ms, 1e-12),
            "speedup_static_route_plus_prepacked_lower_bound": dense_ms / max(static_route_ms + static_prepacked_bmm_ms, 1e-12),
            "ideal_ffn_flop_ratio": candidate_m / D_FF,
            "ideal_ffn_math_speedup": D_FF / candidate_m,
            "nonempty_clusters": int(nonempty.numel()),
            "max_cluster_size": float(nonempty.max().item()) if nonempty.numel() else 0.0,
            "min_cluster_size": float(nonempty.min().item()) if nonempty.numel() else 0.0,
            "imbalance": float((nonempty.max() / nonempty.mean().clamp_min(1.0)).item()) if nonempty.numel() else 0.0,
            "max_exec_diff": max_exec_diff,
            "max_preindexed_diff": max_preindexed_diff,
        }
        rows.append(row)
        print(row)

print(json.dumps(rows, indent=2))

## Optional Checkpoint Quality Oracle

This section is skipped unless `RUN_CHECKPOINT_ORACLE = True` **and** `DENSE_CHECKPOINT` exists on the remote filesystem. It is here for quality/NLL, not for the synthetic speed check above.

In [ ]:
def model_for_dense(config):
    training = dataclasses.replace(config.training, mode="dense_exact")
    model = dataclasses.replace(config.model, topology="dense")
    return dataclasses.replace(config, training=training, model=model)


@torch.no_grad()
def build_svd_factor_cache(dense: DenseModel, *, max_rank: int, device: torch.device):
    factors = []
    for block in dense.blocks:
        weight = block.mlp.wug.weight.detach().float().cpu()
        up_w, gate_w = weight.chunk(2, dim=0)
        layer = {}
        for name, mat in (("up", up_w.t().contiguous()), ("gate", gate_w.t().contiguous())):
            u, s, vh = torch.linalg.svd(mat, full_matrices=False)
            rank = min(max_rank, s.numel())
            layer[f"{name}_a"] = (u[:, :rank] * s[:rank]).to(device=device)
            layer[f"{name}_b"] = vh[:rank, :].to(device=device)
        factors.append(layer)
    return factors


@torch.no_grad()
def cluster_pool_block_output(block, normed, factors, *, rank, clusters, candidate_m):
    flat = normed.reshape(-1, normed.shape[-1])
    q_up = flat @ factors["up_a"][:, :rank]
    q_gate = flat @ factors["gate_a"][:, :rank]
    up_hat = q_up @ factors["up_b"][:rank, :]
    gate_hat = q_gate @ factors["gate_b"][:rank, :]
    gate_act = F.silu(gate_hat)
    wd_rows = block.mlp.wd.weight.t().contiguous()
    wd_norm = block.mlp.wd.weight.detach().norm(dim=0).to(flat.device, flat.dtype)
    scores = (up_hat.abs() + gate_act.abs() + (up_hat * gate_act).abs()) * wd_norm
    assignments = cluster_assignments_from_features(torch.cat([q_up, q_gate], dim=-1), cluster_count=clusters, iters=CLUSTER_ITERS)
    pooled = aggregate_cluster_scores(scores.float(), assignments, cluster_count=min(clusters, flat.shape[0]))
    candidate_ids = torch.topk(pooled, k=min(candidate_m, block.mlp.wd.in_features), dim=-1).indices
    out = flat.new_zeros(flat.shape[0], flat.shape[-1])
    up_weight, gate_weight = block.mlp.wug.weight.detach().chunk(2, dim=0)
    for cluster_idx in range(candidate_ids.shape[0]):
        token_idx = torch.nonzero(assignments == cluster_idx, as_tuple=False).flatten()
        if token_idx.numel() == 0:
            continue
        ids = candidate_ids[cluster_idx]
        xc = flat.index_select(0, token_idx)
        up = xc @ up_weight.index_select(0, ids).t().contiguous()
        gate = xc @ gate_weight.index_select(0, ids).t().contiguous()
        z = up * F.silu(gate)
        out.index_copy_(0, token_idx, z @ wd_rows.index_select(0, ids).contiguous())
    return out.view(*normed.shape[:-1], -1)


@torch.no_grad()
def cluster_pool_model_logits(dense, factors, tokens, *, rank, clusters, candidate_m):
    h = dense.embed(tokens)
    for layer_idx, block in enumerate(dense.blocks):
        u = h + block.attn(block.norm1(h))
        ffn = cluster_pool_block_output(block, block.norm2(u), factors[layer_idx], rank=rank, clusters=clusters, candidate_m=candidate_m)
        h = u + ffn
    hidden = dense.final_norm(h)
    return hidden @ dense.vocab_weight.t(), hidden


if not RUN_CHECKPOINT_ORACLE:
    print("Skipping checkpoint oracle because RUN_CHECKPOINT_ORACLE=False.")
elif not HAVE_REPO_MODEL_CODE:
    print("Skipping checkpoint oracle because repo model imports failed.")
elif not DENSE_CHECKPOINT.exists():
    print("Skipping checkpoint oracle because checkpoint does not exist on remote:", DENSE_CHECKPOINT)
    print("Set DENSE_CHECKPOINT to a real Colab path or export env DENSE_CHECKPOINT before running.")
else:
    config = model_for_dense(load_config(str(CONFIG)))
    if BATCH_SIZE is not None:
        config = dataclasses.replace(config, training=dataclasses.replace(config.training, batch_size=int(BATCH_SIZE)))
    set_seed(SEED)
    streams = load_token_streams(config.data, config.training, config.model.vocab_size)
    dense = DenseModel(dataclasses.replace(config.model, topology="dense")).to(DEVICE)
    payload = torch.load(DENSE_CHECKPOINT, map_location=DEVICE, weights_only=False)
    dense.load_state_dict(payload["model"], strict=True)
    dense.eval()
    factors = build_svd_factor_cache(dense, max_rank=RANK, device=DEVICE)
    tokens_per_batch = config.training.batch_size * config.training.seq_len
    eval_batches = 1 if EVAL_BATCHES is None else int(EVAL_BATCHES)
    batches = streams.eval_batches(config.training)
    quality_rows = []
    for batch_idx in range(eval_batches):
        tokens, targets = next(batches)
        tokens = tokens.to(DEVICE)
        targets = targets.to(DEVICE)
        dense_out = dense(tokens, targets, return_loss_per_sample=True)
        dense_logp = F.log_softmax(dense_out.logits.detach() / TEMPERATURE, dim=-1)
        target_flat = targets.flatten()
        quality_rows.append({"variant": "dense", "batch": batch_idx + 1, "nll": float(dense_out.loss_per_sample.sum().detach().cpu()) / int(targets.numel())})
        for clusters in CLUSTERS:
            for candidate_m in CANDIDATE_M:
                logits, hidden = cluster_pool_model_logits(dense, factors, tokens, rank=RANK, clusters=clusters, candidate_m=candidate_m)
                loss = F.cross_entropy(logits.flatten(0, -2), target_flat, reduction="sum")
                cand_logp = F.log_softmax(logits / TEMPERATURE, dim=-1)
                kl = (dense_logp.exp() * (dense_logp - cand_logp)).sum(dim=-1).sum() * (TEMPERATURE * TEMPERATURE)
                quality_rows.append({
                    "variant": f"cluster_c{clusters}_m{candidate_m}",
                    "batch": batch_idx + 1,
                    "nll": float(loss.detach().cpu()) / int(targets.numel()),
                    "kl_to_dense": float(kl.detach().cpu()) / int(targets.numel()),
                })
    print(json.dumps(quality_rows, indent=2))